# 05 — Serve Qwen with SGLang

SGLang is a serving and inference engine. It is **not** the primary fine-tuning framework in this series: NeMo performs LoRA training, then a deployment-compatible exported model can be served by SGLang.

This notebook prints commands and includes an OpenAI-compatible client. It never starts a server automatically. Practical SGLang use normally requires Linux, a supported NVIDIA CUDA GPU, compatible PyTorch/CUDA packages, model storage, and network access for uncached weights.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_PATH = 'Qwen/Qwen3-1.7B'  # Or a validated, SGLang-compatible exported directory.
HOST = '127.0.0.1'
PORT = 30000
print('Project root:', PROJECT_ROOT)
print('Model:', MODEL_PATH)

## Install on the serving host

Use a dedicated environment or the current SGLang container/instructions rather than adding SGLang to the training environment blindly. A typical Python installation command is:

```bash
python -m pip install --upgrade sglang
```

Select the extras and attention backend recommended by the current [SGLang installation guide](https://docs.sglang.ai/). CUDA/PyTorch compatibility changes more quickly than this learning scaffold.

## Launch the server from a terminal

Start with loopback-only binding while learning:

```bash
python -m sglang.launch_server \
  --model-path Qwen/Qwen3-1.7B \
  --host 127.0.0.1 \
  --port 30000 \
  --dtype bfloat16 \
  --mem-fraction-static 0.80
```

Use `float16` if the target GPU does not support bfloat16. If memory is tight, consult current SGLang quantization options rather than guessing flags. Binding to `0.0.0.0` exposes the service to the host network; add authentication, TLS, and network controls before doing so.

In [ ]:
server_command = [
    'python', '-m', 'sglang.launch_server',
    '--model-path', MODEL_PATH,
    '--host', HOST,
    '--port', str(PORT),
    '--dtype', 'bfloat16',
    '--mem-fraction-static', '0.80',
]
print('Command preview (run in a terminal on the GPU host):')
print(' '.join(server_command))

## Serving the fine-tuned result

A NeMo distributed checkpoint is not automatically an SGLang model directory. Use the export/merge workflow supported by the exact NeMo release, then validate the exported Hugging Face-compatible checkpoint with `transformers` before serving it. SGLang's LoRA-serving flags and supported adapter formats are release-specific; check them before choosing between dynamic LoRA loading and a merged model.

For a first serving smoke test, serve the base model. This separates SGLang/CUDA issues from checkpoint-conversion issues.

In [ ]:
CALL_SERVER = False
BASE_URL = f'http://{HOST}:{PORT}/v1'

if CALL_SERVER:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=os.getenv('SGLANG_API_KEY', 'not-required'))
    response = client.chat.completions.create(
        model=MODEL_PATH,
        messages=[
            {'role': 'system', 'content': 'You are a concise machine learning tutor.'},
            {'role': 'user', 'content': 'Explain the difference between training and serving.'},
        ],
        temperature=0,
        max_tokens=128,
    )
    print(response.choices[0].message.content)
else:
    print('Server call skipped. Start SGLang, verify MODEL_PATH, then set CALL_SERVER=True.')

## Equivalent API request

From another terminal after the server is healthy:

```bash
curl http://127.0.0.1:30000/v1/chat/completions \
  -H 'Content-Type: application/json' \
  -d '{"model":"Qwen/Qwen3-1.7B","messages":[{"role":"user","content":"What is LoRA?"}],"temperature":0,"max_tokens":96}'
```

Check `GET /v1/models` if the request model name is rejected. For production, measure time to first token, tokens per second, concurrency behavior, error rate, and GPU memory under representative traffic.

## Next

Notebook 6 introduces TensorRT-LLM as a more advanced NVIDIA-specific optimization and serving path.